### 1. Environment Setup

#### Importing Libraries

In [1]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn


print(f"Pandas version: {pd.__version__}")
print(f"scikit-learn version: {sklearn.__version__}")

Pandas version: 3.0.5
scikit-learn version: 1.9.0


#### Display Settings

In [2]:
# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

### 2. Data Loading and Schema Overview

#### Load Datasets

In [3]:
# === DATA PATH CONFIGURATION ===
# Local path
DATA_DIR = "./data"

# Verify data directory exists
if os.path.exists(DATA_DIR):
    print(f"✅ Data directory found: {DATA_DIR}")
    print(f"   Files: {os.listdir(DATA_DIR)}")
else:
    print(f"❌ Data directory not found: {DATA_DIR}")
    print("   Please update DATA_DIR to point to your Olist data folder")

✅ Data directory found: ./data
   Files: ['olist_sellers_dataset.csv', '.DS_Store', 'product_category_name_translation.csv', 'olist_orders_dataset.csv', 'olist_order_items_dataset.csv', 'olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_products_dataset.csv']


In [4]:
# Load all Olist tables
def load_olist_data(data_dir):
    """Load all Olist CSV files into a dictionary of DataFrames."""

    tables = {
        'orders': 'olist_orders_dataset.csv',
        'order_items': 'olist_order_items_dataset.csv',
        'customers': 'olist_customers_dataset.csv',
        'products': 'olist_products_dataset.csv',
        'sellers': 'olist_sellers_dataset.csv',
        'payments': 'olist_order_payments_dataset.csv',
        'reviews': 'olist_order_reviews_dataset.csv',
        'geolocation': 'olist_geolocation_dataset.csv',
        'category_translation': 'product_category_name_translation.csv'
    }

    # Date columns to parse
    date_cols = {
        'orders': ['order_purchase_timestamp', 'order_approved_at',
                   'order_delivered_carrier_date', 'order_delivered_customer_date',
                   'order_estimated_delivery_date'],
        'order_items': ['shipping_limit_date'],
        'reviews': ['review_creation_date', 'review_answer_timestamp']
    }

    data = {}
    for name, filename in tables.items():
        filepath = os.path.join(data_dir, filename)
        parse_dates = date_cols.get(name, None)
        data[name] = pd.read_csv(filepath, parse_dates=parse_dates)
        print(f"Loaded {name}: {data[name].shape[0]:,} rows × {data[name].shape[1]} cols")

    return data

# Load data
olist = load_olist_data(DATA_DIR)

Loaded orders: 99,441 rows × 8 cols
Loaded order_items: 112,650 rows × 7 cols
Loaded customers: 99,441 rows × 5 cols
Loaded products: 32,951 rows × 9 cols
Loaded sellers: 3,095 rows × 4 cols
Loaded payments: 103,886 rows × 5 cols
Loaded reviews: 99,224 rows × 7 cols
Loaded geolocation: 1,000,163 rows × 5 cols
Loaded category_translation: 71 rows × 2 cols


#### Schema Exploration - Overview of all tables

In [5]:
# Quick schema overview
def schema_summary(data_dict):
    """Generate a summary of all tables."""
    summary = []
    for name, df in data_dict.items():
        summary.append({
            'Table': name,
            'Rows': f"{df.shape[0]:,}",
            'Columns': df.shape[1],
            'Memory (MB)': f"{df.memory_usage(deep=True).sum() / 1e6:.2f}",
            'Columns List': ', '.join(df.columns[:5]) + ('...' if len(df.columns) > 5 else '')
        })
    return pd.DataFrame(summary)

schema_summary(olist)

,Table,Rows,Columns,Memory (MB),Columns List
0,orders,"99,441",8,25.85,"order_id, customer_id, order_status, order_pur..."
1,order_items,"112,650",7,30.98,"order_id, order_item_id, product_id, seller_id..."
2,customers,"99,441",5,27.88,"customer_id, customer_unique_id, customer_zip_..."
3,products,"32,951",9,6.60,"product_id, product_category_name, product_nam..."
4,sellers,"3,095",4,0.62,"seller_id, seller_zip_code_prefix, seller_city..."
5,payments,"103,886",5,17.02,"order_id, payment_sequential, payment_type, pa..."
6,reviews,"99,224",7,29.12,"review_id, order_id, review_score, review_comm..."
7,geolocation,"1,000,163",5,135.67,"geolocation_zip_code_prefix, geolocation_lat, ..."
8,category_translation,71,2,0.01,"product_category_name, product_category_name_e..."


### 3. Orders Table Exploration and Missingness Analysis

#### Orders Table - Sample

In [6]:
# First 5 rows of Orders Table
orders = olist['orders']
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


#### Shape and Summary

In [7]:
# Orders Table Summary
print("=== Shape ===")
print(f"{orders.shape}")
print("\n")
print("=== Summary ===")
orders.info()

=== Shape ===
(99441, 8)


=== Summary ===
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


#### Order_Status Breakdown

In [8]:
print("=== Order Status Breakdown ===")
status_counts = orders['order_status'].value_counts()
status_pct = orders['order_status'].value_counts(normalize=True) * 100
print(pd.DataFrame({'Count': status_counts, 'Percentage (%)': status_pct.round(2)}))

=== Order Status Breakdown ===
              Count  Percentage (%)
order_status                       
delivered     96478           97.02
shipped        1107            1.11
canceled        625            0.63
unavailable     609            0.61
invoiced        314            0.32
processing      301            0.30
created           5            0.01
approved          2            0.00


#### Missingness Breakdown

In [9]:
print("=== Columns with Missing Value ===")
null_dates = olist['orders'].isnull().sum()
print(null_dates[null_dates > 0])

=== Columns with Missing Value ===
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64


#### Missingness: `order_delivered_customer_date`

In [10]:
# identifying rows where 'order_delivered_customer_date' is empty - returns True/False
is_blank = orders['order_delivered_customer_date'].isnull()

# filter to keep only rows where is_blank == True
blank_orders = orders[is_blank]

# group by order_status
blank_orders.groupby('order_status')['order_id'].count()

order_status
approved          2
canceled        619
created           5
delivered         8
invoiced        314
processing      301
shipped        1107
unavailable     609
Name: order_id, dtype: int64

##### Analysis - Why are there missing values in `order_delivered_customer_date`?

- 622 Orders (Status = Approved, Created, Invoiced, Processing (2 + 5 + 314 + 301)) are still in early warehouse fulfillment stage. They have not yet been dispatched yet.
- 619 Orders (Status = Canceled) are cancelled which means delivery never took place.
- 609 Orders (Status = Unavailable) are never fulfilled as the product is out of stock. No delivery took place.
- 1107 Orders (Status = Shipped) are in the midst of delivery. They have not yet reached customers
- 8 Orders (Status = Delivered) are orders with actual missing information. Need to handle these records. 

#### Missingness: `order_delivered_carrier_date`

In [11]:
# identifying rows where 'order_delivered_carrier_date' is empty - returns True/False
is_blank = orders['order_delivered_carrier_date'].isnull()

# filter to keep only rows where is_blank == True
blank_orders = orders[is_blank]

# group by order_status
blank_orders.groupby('order_status')['order_id'].count()

order_status
approved         2
canceled       550
created          5
delivered        2
invoiced       314
processing     301
unavailable    609
Name: order_id, dtype: int64

##### Analysis - Why are there missing values in `order_delivered_carrier_date`?

- 622 Orders (Status = Approved, Created, Invoiced, Processing (2 + 5 + 314 + 301)) are still in early warehouse fulfillment stage. They have not yet been dispatched yet.
- 550 Orders (Status = Canceled) are cancelled which means delivery never took place. Notice this count is different (lesser) from the count in 'order_delivered_customer_date`. A possible reason is that the cancellation came after seller dropped off the product to the carrier (but before reaching the customer).
- 609 Orders (Status = Unavailable) are never fulfilled as the product is out of stock. No delivery took place.
- 2 Orders (Status = Delivered) are orders with actual missing information. Need to handle these records. 

#### Analysis: Overall
- **Pre-dispatch Cancellations (550 Orders):** Cancelled prior to courier handover. Both `order_delivered_carrier_date` and `order_delivered_customer_date` timestamps are missing.
- **In-transit Cancellations (69 Orders):** Difference ($619 - 50 = 69$) represents packages dispatched to carrier but cancelled before reaching the buyer.
- **Active Shipments (1,107 Orders):** Dispatched to carrier but still in transit. These have `order_delivered_carrier_date` timestamps but do not have `order_delivered_customer_date` timestamps.
- **Actual Missing Information (2 to 8 Orders):** These are completed deliveries that have missing `order_delivered_carrier_date` or `order_delivered_customer_date` or both.

#### Implications and Filtering Decision
- Because target variable for **Delivery Performance** requires measuring actual lead time from purchase to delivery ($\text{Delivered Date} - \text{Purchase Date}$), orders lacking a customer delivery date cannot be included.
- **Filtering Rule:** `order_status == 'delivered'` and `order_delivered_customer_date` is not null.
- **Rows retained:** 96,470 orders (need to exclude the 8 in the missingness analysis) which is about 97% of the total orders.

In [12]:
# Applying Filtering Rule
# Conditions
is_delivered = orders['order_status'] == 'delivered'
has_delivery_date = orders['order_delivered_customer_date'].notna()

# Filtering
valid_orders_filter = is_delivered & has_delivery_date
delivered_orders = orders[valid_orders_filter].copy()

# Summary/Count
total_count = len(orders)
retained_count = len(delivered_orders)
dropped_count = total_count - retained_count

print(f"Total raw orders: {total_count:,}")
print(f"Retained delivered orders: {retained_count:,} ({retained_count/total_count*100:.2f}%)")
print(f"Filtered out: {dropped_count:,} ({ dropped_count/total_count*100:.2f}%)")

Total raw orders: 99,441
Retained delivered orders: 96,470 (97.01%)
Filtered out: 2,971 (2.99%)
